# 03 - Baselines

Question: how well can simple, no-learning guesses predict next-composite LST? Any real model has to beat these.

Split decided in 02: train on 2003-2018, test on 2019-2023.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/dataset.parquet')
df['comp'] = df.groupby(['pid', 'year']).cumcount()
df = df.dropna(subset=['lst_target'])

train = df[df['year'] <= 2018]
test = df[df['year'] >= 2019]
print(len(train), len(test))
print(train['year'].min(), train['year'].max(), test['year'].min(), test['year'].max())


310577 99840
2003 2018 2019 2023


In [2]:
def rmse(y, pred):
    return float(np.sqrt(np.mean((y - pred)**2)))

def mae(y, pred):
    return float(np.mean(np.abs(y - pred)))

t = test.dropna(subset=['lst_prev'])
print(len(t))
print(f"persistence  RMSE {rmse(t['lst_target'], t['lst_prev']):.2f}   MAE {mae(t['lst_target'], t['lst_prev']):.2f}")

84777
persistence  RMSE 4.91   MAE 3.83


In [3]:
clim = train.groupby(['pid', 'comp'])['lst_target'].mean().rename('clim')
tc = test.join(clim, on=['pid', 'comp']).dropna(subset=['clim'])
print(len(tc))
print(f"climatology  RMSE {rmse(tc['lst_target'], tc['clim']):.2f}   MAE {mae(tc['lst_target'], tc['clim']):.2f}")

same = tc.dropna(subset=['lst_prev'])
print(len(same))
print(f"same rows:  persistence {rmse(same['lst_target'], same['lst_prev']):.2f}   climatology {rmse(same['lst_target'], same['clim']):.2f}")

99838
climatology  RMSE 4.34   MAE 3.44
84777
same rows:  persistence 4.91   climatology 4.39


In [4]:
prev_clim = clim.rename('prev_clim').reset_index()
prev_clim['comp'] = prev_clim['comp'] + 1

def prep(d):
    d = d.join(clim, on=['pid', 'comp'])
    d = d.merge(prev_clim, on=['pid', 'comp'], how='left')
    d['prev_anom'] = d['lst_prev'] - d['prev_clim']
    d['anom'] = d['lst_target'] - d['clim']
    return d.dropna(subset=['clim', 'prev_anom'])

tr, te = prep(train), prep(test)
print(len(tr), len(te))

k = (tr['prev_anom'] * tr['anom']).sum() / (tr['prev_anom'] ** 2).sum()
print(round(k, 3))

print('climatology      ', round(rmse(te['lst_target'], te['clim']), 3))
print('persistence      ', round(rmse(te['lst_target'], te['lst_prev']), 3))
print('clim + k*prev    ', round(rmse(te['lst_target'], te['clim'] + k * te['prev_anom']), 3))


231884 76310
0.287
climatology       4.443
persistence       5.04
clim + k*prev     4.211


Spread of the data (guess the mean)	5.06	The no-skill line

Persistence	5.04	The raw last composite is a noisy guide

Climatology	4.44	Place and season carry most of the signal

Climatology + 0.29 × last departure	4.21	The previous composite's departure adds a little